# Day 10: What is the model looking at? (Grad-CAM)
Grad-CAM paints a heatmap over an image showing which regions pushed the model toward its answer. Red/yellow = strong influence, blue = little.

The question for a leaf-disease model: **is it looking at the lesions and streaks, or at something it shouldn't (soil, hands, the image border, leaf edges, a uniform patch)?** If it keys on background, high accuracy is partly a shortcut, and that belongs in your write-up as a finding.

**Rules for using the test set here:** this is analysis only. The test mistakes are shown because they are the thing we want to understand, but nothing you see should be used to retrain, tune or re-select a model. If a finding makes you want to change something, do it as a new, clearly labelled experiment judged on the validation set, and say so in the write-up.

What it makes: for each class, 4 correctly-classified validation images; and for the 12 most confident test mistakes, two heatmaps each: what drove the **wrong** answer, and what the model saw as evidence for the **true** class. A deletion test then checks, on 60 more validation images, whether the heatmaps point at regions the model really depends on. Figures are saved to `maize_ckpt/figures` on Drive, ready for the README.

Use a GPU runtime (Runtime > Change runtime type > T4 GPU); CPU works but is slower.

## 1. Setup

In [ ]:
import os, zipfile
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import torch, torchvision
from torch import nn
from torchvision import datasets, transforms
from PIL import Image

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)
from google.colab import drive
drive.mount("/content/drive")
CKPT_DIR = "/content/drive/MyDrive/maize_ckpt"
FIG_DIR = CKPT_DIR + "/figures"
os.makedirs(FIG_DIR, exist_ok=True)
PRIMARY = "mnv2_last6"          # the model declared before the test run

ROOT = "/content/maize_split"
if not os.path.isdir(ROOT + "/test"):
    z = "/content/drive/MyDrive/maize_split.zip"
    assert os.path.exists(z), "No /content/maize_split and no maize_split.zip on Drive: re-run Day 3 (and its optional Drive cell)."
    os.makedirs(ROOT, exist_ok=True)
    with zipfile.ZipFile(z) as zf:
        zf.extractall(ROOT)
print("ready")


## 2. Model and Grad-CAM
Grad-CAM uses the last convolutional layer (the last layer that still has a spatial map, 7 x 7 for a 224-pixel image), weights each of its channels by how much it affects the chosen class, and sums them. The 7 x 7 map is stretched over the image, so it is coarse by design: it can say "this region", not "this exact pixel".

In [ ]:
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
eval_tf = transforms.Compose([transforms.Resize(256), transforms.CenterCrop(224), transforms.ToTensor(), transforms.Normalize(MEAN, STD)])
mean_t = torch.tensor(MEAN).view(3, 1, 1); std_t = torch.tensor(STD).view(3, 1, 1)
val_ds = datasets.ImageFolder(ROOT + "/val", eval_tf)
test_ds = datasets.ImageFolder(ROOT + "/test", eval_tf)
classes = val_ds.classes

def build_model(arch):
    if arch == "resnet18":
        m = torchvision.models.resnet18(weights=None); m.fc = nn.Linear(m.fc.in_features, len(classes))
    else:
        m = torchvision.models.mobilenet_v2(weights=None); m.classifier[1] = nn.Linear(m.classifier[1].in_features, len(classes))
    return m

ck = torch.load(f"{CKPT_DIR}/phase2_{PRIMARY}_best.pt", map_location=device)
assert ck["classes"] == classes
m = build_model(ck["arch"]).to(device); m.load_state_dict(ck["model"]); m.eval()
target_layer = m.features[-1] if ck["arch"] == "mobilenet_v2" else m.layer4[-1]

class GradCAM:
    def __init__(self, model, layer):
        self.model, self.acts, self.grads = model, None, None
        layer.register_forward_hook(self._fwd)
    def _fwd(self, mod, inp, out):
        self.acts = out.detach()
        if out.requires_grad:        # skipped when the model runs under torch.no_grad() (e.g. the deletion test)
            out.register_hook(lambda g: setattr(self, "grads", g.detach()))
    def __call__(self, x, cls=None):
        """Heatmaps for the predicted class, or for the class indices given in `cls`."""
        self.model.zero_grad()
        out = self.model(x)
        idx = out.argmax(1) if cls is None else cls
        out.gather(1, idx.view(-1, 1)).sum().backward()
        w = self.grads.mean((2, 3), keepdim=True)
        cam = torch.relu((w * self.acts).sum(1))
        cam = cam / (cam.flatten(1).max(1)[0].view(-1, 1, 1) + 1e-8)
        return cam.cpu(), torch.softmax(out.detach(), 1).cpu()
gc = GradCAM(m, target_layer)

def prep(paths):
    return torch.stack([eval_tf(Image.open(p).convert("RGB")) for p in paths])

def overlay(img_t, cam, alpha=0.45):
    img = (img_t.cpu() * std_t + mean_t).clamp(0, 1).permute(1, 2, 0).numpy()
    c = torch.nn.functional.interpolate(cam[None, None], size=img.shape[:2], mode="bilinear", align_corners=False)[0, 0].numpy()
    return img, (1 - alpha) * img + alpha * plt.cm.jet(c)[..., :3]
print("model:", PRIMARY, "| Grad-CAM layer:", type(target_layer).__name__)


## 3. Correct predictions, one figure per class
Top row: what the model was shown (the centre crop). Bottom row: where it looked. For each class ask: is the heat on the symptom (streaks, necrotic patches, lesions), or somewhere that would be there whether or not the leaf were diseased?

In [ ]:
rng = np.random.default_rng(0)
vt = np.array(val_ds.targets)
for ci, cname in enumerate(classes):
    cand = np.where(vt == ci)[0]; rng.shuffle(cand); cand = cand[:12]
    x = prep([val_ds.samples[i][0] for i in cand]).to(device)
    cam, pr = gc(x)
    ok = [k for k in range(len(cand)) if int(pr[k].argmax()) == ci][:4]
    fig, ax = plt.subplots(2, len(ok), figsize=(3.2 * len(ok), 6.6), squeeze=False)
    for j, k in enumerate(ok):
        img, ov = overlay(x[k], cam[k])
        ax[0][j].imshow(img); ax[1][j].imshow(ov)
        ax[0][j].set_title(f"{os.path.basename(val_ds.samples[cand[k]][0])}\n{cname}, confidence {pr[k].max():.2f}", fontsize=8)
        ax[0][j].axis("off"); ax[1][j].axis("off")
    plt.suptitle(f"{cname}: correct predictions (top: input, bottom: Grad-CAM)")
    plt.tight_layout()
    plt.savefig(f"{FIG_DIR}/gradcam_correct_{cname}.png", dpi=150, bbox_inches="tight")
    plt.show()


## 4. The most confident test mistakes
Three rows per image: the input; where the model looked to reach its **wrong** answer; where the evidence for the **true** class was. If the two heatmaps highlight the same region, the model saw the same patch and read it differently (a genuinely ambiguous symptom). If they differ, it attended to something else entirely.

In [ ]:
errs = pd.read_csv(f"{CKPT_DIR}/test_errors_{PRIMARY}.csv")
path_of = {os.path.basename(p): p for p, _ in test_ds.samples}
cls_idx = {c: i for i, c in enumerate(classes)}
sel = errs.head(12)
print(sel.to_string(index=False))
for start in range(0, len(sel), 4):
    part = sel.iloc[start:start + 4]
    x = prep([path_of[f] for f in part.file]).to(device)
    pred_i = torch.tensor([cls_idx[c] for c in part.predicted], device=device)
    true_i = torch.tensor([cls_idx[c] for c in part["true"]], device=device)
    cam_p, pr = gc(x, pred_i)
    cam_t, _ = gc(x, true_i)
    fig, ax = plt.subplots(3, len(part), figsize=(3.2 * len(part), 9.6), squeeze=False)
    for j in range(len(part)):
        r = part.iloc[j]
        img, ov_p = overlay(x[j], cam_p[j]); _, ov_t = overlay(x[j], cam_t[j])
        ax[0][j].imshow(img); ax[1][j].imshow(ov_p); ax[2][j].imshow(ov_t)
        ax[0][j].set_title(f"{r['file']}\ntrue {r['true']}, predicted {r['predicted']} ({r['confidence']:.2f})", fontsize=8)
        ax[1][j].set_title(f"drove the wrong answer ({r['predicted']})", fontsize=8)
        ax[2][j].set_title(f"evidence for the true class ({r['true']})", fontsize=8)
        for a in ax[:, j]: a.axis("off")
    plt.tight_layout()
    plt.savefig(f"{FIG_DIR}/gradcam_mistakes_{start // 4 + 1}.png", dpi=150, bbox_inches="tight")
    plt.show()
print("figures saved in", FIG_DIR, ":", sorted(os.listdir(FIG_DIR)))


## 5. Do the hot regions really matter to the model? (deletion test, version 2)
A heatmap can look convincing and still be wrong about what the model uses. This test checks it on 60 correctly classified **validation** images (20 per class). The image is split into the same 7 x 7 grid the heatmap lives on, and **10 of the 49 cells (about 20% of the picture)** are greyed out in three different ways, then the model's confidence in its answer is measured:

- **hot:** the 10 cells the heatmap rates hottest;
- **random:** 10 cells chosen at random (averaged over 3 draws);
- **cold:** the 10 cells the heatmap rates coolest.

If the heatmap is telling the truth, confidence should fall most when the hot cells are hidden and least when the cold cells are hidden (hot < random < cold). Using blocks for all three keeps the comparison fair: an earlier version of this test scattered random single pixels, which damages fine leaf texture everywhere and is not comparable to hiding one region.

This says nothing about *whether those regions are lesions*; that stays your visual judgement.

In [ ]:
NC = 10                                      # cells hidden out of the 7 x 7 = 49 (20.4% of the image)

def conf_after(x, mask, ci):
    """Model's probability for class ci after greying out the masked pixels (0 = average colour after normalisation)."""
    xm = x.masked_fill(mask[:, None].to(device), 0.0)
    with torch.no_grad():
        return torch.softmax(m(xm), 1)[torch.arange(len(x)), ci].cpu()

def block_mask(cell_idx, n):
    """cell_idx: (n, NC) indices into the 49 grid cells -> (n, 224, 224) boolean mask made of 32x32-pixel blocks."""
    mk = torch.zeros(n, 49, dtype=torch.bool)
    mk[torch.arange(n)[:, None], cell_idx] = True
    return mk.view(n, 7, 7).float().repeat_interleave(32, 1).repeat_interleave(32, 2) > 0

rng = np.random.default_rng(1); torch.manual_seed(1)
vt = np.array(val_ds.targets)
rows = []
for ci, cname in enumerate(classes):
    cand = np.where(vt == ci)[0]; rng.shuffle(cand)
    x = prep([val_ds.samples[i][0] for i in cand[:40]]).to(device)
    cam, pr = gc(x)                                   # cam: (n, 7, 7)
    ok = [k for k in range(len(x)) if int(pr[k].argmax()) == ci][:20]
    x, cam, pr = x[ok], cam[ok], pr[ok]
    n = len(x)
    flat = cam.flatten(1)
    p0 = pr[torch.arange(n), ci]
    p_hot = conf_after(x, block_mask(flat.topk(NC, dim=1).indices, n), ci)
    p_cold = conf_after(x, block_mask(flat.topk(NC, dim=1, largest=False).indices, n), ci)
    p_rand = torch.stack([conf_after(x, block_mask(torch.rand(n, 49).argsort(1)[:, :NC], n), ci) for _ in range(3)]).mean(0)
    for j in range(n):
        rows.append({"class": cname, "original": float(p0[j]), "hot hidden": float(p_hot[j]), "random hidden": float(p_rand[j]), "cold hidden": float(p_cold[j]),
                     "flipped by hot": int(p_hot[j] < 0.5), "flipped by random": int(p_rand[j] < 0.5), "flipped by cold": int(p_cold[j] < 0.5)})
dele = pd.DataFrame(rows)
cols = ["original", "hot hidden", "random hidden", "cold hidden", "flipped by hot", "flipped by random", "flipped by cold"]
by_class = dele.groupby("class")[cols].mean().round(3)
tot = dele[cols].mean().round(3)
print(f"Average confidence in the model's answer when {NC} of 49 grid cells are hidden, and the share of images whose answer flips (confidence < 0.5):")
print(by_class)
print(f"\nall {len(dele)} images:"); print(tot.to_string())
faithful = bool(tot["hot hidden"] < tot["random hidden"] < tot["cold hidden"])
print("\nheatmap passes (hot < random < cold):", faithful)


## 6. What to write down
For each figure, jot a line or two:
- **Lesions or symptoms highlighted** (good): the model is reading the disease.
- **Leaf edges, the midrib, or a plain smooth area highlighted:** it may be using leaf structure or lighting rather than symptoms.
- **Soil, hands, the image border or a corner highlighted** (bad): a shortcut. Tell me how many of the images do this, out of how many you looked at.
- **Diffuse heat everywhere:** common for MSV, where streaks run across the whole leaf; not a problem by itself.
- For the mistakes: did the wrong-answer and true-class maps overlap?

Be honest about what you can't tell: Grad-CAM is a rough guide, and for a 7 x 7 map "near a lesion" is often the most it can say. Paste your notes into the chat, and I'll help turn them into the write-up paragraph.

## 7. Save a summary
Paste `day10_summary.md` back into the chat together with your notes. The heatmap figures are already on your Drive in `maize_ckpt/figures`, ready for the README.

In [ ]:
lines = [f"# Day 10 summary (Grad-CAM, {PRIMARY})",
         f"- explained layer: {type(target_layer).__name__} (7x7 grid); correct examples from the validation set, the 12 most confident mistakes from the test set",
         f"- deletion test on {len(dele)} correct validation images, hiding {NC} of 49 grid cells (about 20%): mean confidence {tot['original']:.3f} originally; {tot['hot hidden']:.3f} with the hot cells hidden, {tot['random hidden']:.3f} random cells, {tot['cold hidden']:.3f} cold cells",
         f"- answer flipped (confidence < 0.5): hot {tot['flipped by hot']:.3f}, random {tot['flipped by random']:.3f}, cold {tot['flipped by cold']:.3f}; heatmap passes (hot < random < cold): {faithful}",
         "- by class (confidence after hiding hot / random / cold): " + "; ".join(f"{c} {by_class.loc[c, 'hot hidden']:.3f} / {by_class.loc[c, 'random hidden']:.3f} / {by_class.loc[c, 'cold hidden']:.3f}" for c in classes),
         f"- figures in {FIG_DIR}: {', '.join(sorted(os.listdir(FIG_DIR)))}",
         "- notes (fill in): where is the heat on correct examples? lesions or shortcuts (soil, hands, border)? how many of how many? what about the mistakes?"]
open("/content/day10_summary.md", "w").write("\n".join(lines) + "\n")
print("\n".join(lines))
try:
    from google.colab import files
    files.download("/content/day10_summary.md")
except ImportError:
    pass
